In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time

In [3]:
g = lambda x: x**2
G = lambda x: x**3 / 3
p = lambda x: 6*x**3 + 8*x
P = lambda x: 1.5*x**4 + 4*x**2
h = lambda x: x**5 + 7*x + 3
H = lambda x: x**6/6 + 3.5*x**2 + 3*x

functions = (("g", g, G), ("p", p, P), ("h", h, H))

Die folgenden Funktionen definieren die drei äquidistanten Quadraturregeln, die wir gesehen haben: 

In [4]:
def mittelpunkt(f, a, b):
    return f((b + a) / 2) * (b - a)

def trapez(f, a, b):
    return 0.5 * (f(a) + f(b)) * (b - a)

def simpson(f, a, b): 
    return (b - a) * (f(a) + f(b) + 4 * f((a + b) / 2)) / 6

Im Folgenden probieren wir die Quadraturformeln der drei Regeln direkt aus. Wir sehen, dass bei Polynomen zweiten und dritten Grades die Simpson-Regel noch exakt ist. (`p`und `g`)

In [5]:
a, b = 0, 4

for name, f, F in functions: 
    print(f"Function {name}:")
    print(f"Integral from {a} to {b}: {F(b)-F(a)}")
    print(f"\tmittelpunktregel: {mittelpunkt(f, a, b)}")
    print(f"\ttrapezregel: {trapez(f, a, b)}")
    print(f"\tsimpsonregel: {simpson(f, a, b)}")

Function g:
Integral from 0 to 4: 21.333333333333332
	mittelpunktregel: 16.0
	trapezregel: 32.0
	simpsonregel: 21.333333333333332
Function p:
Integral from 0 to 4: 448.0
	mittelpunktregel: 256.0
	trapezregel: 832.0
	simpsonregel: 448.0
Function h:
Integral from 0 to 4: 750.6666666666666
	mittelpunktregel: 196.0
	trapezregel: 2116.0
	simpsonregel: 836.0


## Summierte Regeln
Um bei $h$ etwas näher ans Resultat zu kommen, können wir z.B. eine summierte Simpson-Regel anwenden, um die Annäherung zu verbessern. 
Wir verwenden also die Simpson-Regel (vektorisiert) auf alle Subintervalle $[x_i, x_i+1]$ und summieren diese auf. 

In [6]:
n = 20

def simpson_naiv(f, a, b, n):
    x = np.linspace(a, b, n+1)
    s = simpson(f, x[:-1], x[1:])
    return np.sum(s)

approx = simpson_naiv(h, a, b, n)

print(f"Integral:\t{H(b) - H(a)}")
print(f"Annäherung:\t{approx}")

Integral:	750.6666666666666
Annäherung:	750.6672


Wir sehen hier schon eine starke Verbesserung. Mit höherem $n$ wird diese Annäherung zunehmend genauer (Wie Riemann-Summen in Analysis). 

### Effizienz
Hier merken wir allerdings, dass wir viele Operationen mehrfach durchführen - Wir werten $x_2,...,x_{N-1}$ jeweils zweimal aus für beide benachbarten Intervalle, und teilen $n$-mal durch $6n$. <br>
Es lohnt sich dementsprechend, die Mathematik als typische Quadraturformel mit $c_j$ und $w_j$ umzuschreiben: 

In [42]:
def simpson_summiert(f, a, b, n):
    x = np.linspace(a, b, 2*n+1)    # c_j
    y = f(x)

    w = np.ones(2*n+1)              # w_j
    w[1:-1:2] *= 4  # Mittelpunkte
    w[2:-2:2] *= 2  # Breakpoints

    return np.sum(y * w) / 6 / n * (b-a)

In [35]:
approx = simpson_summiert(h, a, b, n)
print(f"Integral:\t{H(b) - H(a)}")
print(f"Annäherung:\t{approx}")

Integral:	750.6666666666666
Annäherung:	750.6666666666666


Das Resultat hier ist gleich, aber wir teilen nur einmal durch $6n$ und werten $f$ pro Stelle nur einmal aus. 

In [43]:
n = 2**24
print(f"Laufzeit für {n} Stützstellen: ")

start = time.time()
approx = simpson_naiv(h, a, b, n)
end = time.time()
print(f"Langsam summierte Simpson-Regel:\t{end - start}s")

start = time.time()
approx = simpson_summiert(h, a, b, n)
end = time.time()
print(f"Schnell summierte Simpson-Regel:\t{end - start}s")

Laufzeit für 16777216 Stützstellen: 
Langsam summierte Simpson-Regel:	1.523979663848877s
Schnell summierte Simpson-Regel:	1.1874773502349854s


Wir sehen: Die Optimierung ist nicht riesig. Es kann sich aber lohnen, eine summierte Quadraturregel auf Gewichte und Stützstellen im ganzen Intervall $[a,b]$ umzuschreiben. 